In [3]:
import sys
import os

# Add the parent directory (src) to the system path
# The '..' tells it to look one folder up from where the notebook is currently running
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [11]:
from sqlitesearch import VectorSearchIndex
from sentence_transformers import SentenceTransformer



model = SentenceTransformer('all-MiniLM-L6-v2')

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [14]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5, filter_dict={"course": "llm-zoomcamp"})

In [15]:
results

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

### Using With RAG

In [20]:
from rag_helper import RAGBase

class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}
        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [21]:
from client import client


assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=client
)

In [24]:
assistant.rag("How can I join LLM Zoomcamp?")

"You can join the LLM Zoomcamp without needing a confirmation email. You're accepted automatically after registering, and you can start learning and submitting homework while the registration form is open. There’s no requirement to be checked against a registered list."

In [25]:
vs_index.close()